In [ ]:
from __future__ import annotations

import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import Image, display

PROJECT_ROOT = Path.cwd().resolve().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from scripts.eda_sliding_electric import (
    ANOMALY_NOTES,
    YEARS,
    build_engine,
    build_sliding_frame,
    electric_meter_config,
    fetch_weather_data,
    load_electric_meter_data,
    figure_to_png_bytes,
    print_anomaly_notes,
    render_sliding_plot,
    resolve_feature_columns,
    summarize_sliding_by_year,
)

engine = build_engine()
weather_df = fetch_weather_data(engine).copy()
weather_df['ts'] = pd.to_datetime(weather_df['ts'], utc=True, errors='coerce')
for column in ['Ta', 'Igm']:
    if column in weather_df.columns:
        weather_df[column] = pd.to_numeric(weather_df[column], errors='coerce')

raw_meter_data = {}
resolved_meter_features = {}
for meter_urn, configured_features in electric_meter_config.items():
    try:
        df = load_electric_meter_data(engine, weather_df, meter_urn)
    except Exception as exc:
        print(f'{meter_urn}: 데이터 로드 실패 - {exc}')
        continue
    raw_meter_data[meter_urn] = df
    resolved_meter_features[meter_urn] = resolve_feature_columns(df, configured_features)

YEARS


In [ ]:
for meter_urn, features in resolved_meter_features.items():
    df = raw_meter_data[meter_urn]
    if not features:
        print(f'{meter_urn}: 유효 컬럼 없음, skip')
        continue

    for feature in features:
        base_sliding_df = build_sliding_frame(df, feature)
        if base_sliding_df.empty or base_sliding_df[feature].dropna().empty:
            print(f'{meter_urn} - {feature}: 유효 데이터 없음, skip')
            continue

        sliding_df = base_sliding_df.copy()
        fig = render_sliding_plot(meter_urn, feature, sliding_df)
        display(Image(data=figure_to_png_bytes(fig)))
        plt.close(fig)


In [ ]:
for meter_urn, features in resolved_meter_features.items():
    df = raw_meter_data[meter_urn]
    for feature in features:
        base_sliding_df = build_sliding_frame(df, feature)
        if base_sliding_df.empty or base_sliding_df[feature].dropna().empty:
            continue
        sliding_df = base_sliding_df.copy()
        stats_df = summarize_sliding_by_year(sliding_df, meter_urn, feature)
        if stats_df.empty:
            continue
        print(f'[{meter_urn} — {feature}]')
        display(stats_df[['year', 'mean', 'std', 'min', 'max', 'null_ratio']].set_index('year'))


In [ ]:
print_anomaly_notes()
